# SQL CTEs — Runnable Examples

SQLite examples with fictional data. Run the notebook from top to bottom.

In [1]:
import sqlite3
import pandas as pd
connection = sqlite3.connect(':memory:')
print('SQLite ready')

SQLite ready


## 1. Create sample tables

In [2]:
connection.executescript('''
CREATE TABLE departments (department_id INTEGER PRIMARY KEY, department_name TEXT, location TEXT);
INSERT INTO departments VALUES (1,'Engineering','Indore'),(2,'Sales','Pune'),(3,'HR','Mumbai'),(4,'Finance','Indore'),(5,'Research','Pune');
CREATE TABLE employees (employee_id INTEGER PRIMARY KEY, employee_name TEXT, department_id INTEGER, salary REAL, location TEXT);
INSERT INTO employees VALUES
(1,'Asha',1,72000,'Indore'),(2,'Ravi',1,68000,'Pune'),(3,'Meera',1,61000,'Indore'),
(4,'Kabir',2,85000,'Pune'),(5,'Neha',2,52000,'Mumbai'),(6,'Arjun',2,48000,'Indore'),
(7,'Isha',3,78000,'Mumbai'),(8,'Dev',3,42000,'Pune'),(9,'Tara',4,65000,'Indore'),
(10,'Omar',4,58000,'Mumbai'),(11,'Zoya',4,63000,'Pune'),(12,'Nikhil',4,NULL,'Indore');
CREATE TABLE customers (customer_id INTEGER PRIMARY KEY, customer_name TEXT);
INSERT INTO customers VALUES (1,'Customer A'),(2,'Customer B'),(3,'Customer C'),(4,'Customer D'),(5,'Customer E'),(6,'Customer F'),(7,'Customer G');
CREATE TABLE orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT, amount REAL, status TEXT);
INSERT INTO orders VALUES
(101,1,'2026-01-10',250,'completed'),(102,1,'2026-01-12',125,'completed'),
(103,2,'2026-01-15',600,'cancelled'),(104,3,'2026-02-01',80,'pending'),
(105,5,'2026-02-03',320,'completed'),(106,2,'2026-02-08',180,'completed');
CREATE TABLE sales (sale_id INTEGER PRIMARY KEY, product TEXT, quantity INTEGER, unit_price REAL);
INSERT INTO sales VALUES (1,'Laptop',2,650),(2,'Mouse',10,20),(3,'Laptop',1,650),(4,'Keyboard',4,45),(5,'Mouse',5,20);
''')
print('Sample tables created')

Sample tables created


## 2. Basic CTE

In [3]:
pd.read_sql_query('WITH high_earners AS (SELECT employee_name, department_id, salary FROM employees WHERE salary >= 60000) SELECT * FROM high_earners ORDER BY salary DESC', connection)

,employee_name,department_id,salary
0,Kabir,2,85000.0
1,Isha,3,78000.0
2,Asha,1,72000.0
3,Ravi,1,68000.0
4,Tara,4,65000.0
5,Zoya,4,63000.0
6,Meera,1,61000.0


## 3. CTE with aggregation

In [4]:
pd.read_sql_query('WITH department_stats AS (SELECT department_id, COUNT(*) AS employee_count, AVG(salary) AS average_salary FROM employees GROUP BY department_id) SELECT * FROM department_stats WHERE average_salary > 60000', connection)

,department_id,employee_count,average_salary
0,1,3,67000.000000
1,2,3,61666.666667
2,4,4,62000.000000


## 4. Multiple CTEs and a join

In [5]:
pd.read_sql_query('WITH department_avg AS (SELECT department_id, AVG(salary) AS average_salary FROM employees GROUP BY department_id), high_departments AS (SELECT * FROM department_avg WHERE average_salary > 60000) SELECT d.department_name, h.average_salary FROM high_departments h JOIN departments d ON d.department_id = h.department_id', connection)

,department_name,average_salary
0,Engineering,67000.000000
1,Sales,61666.666667
2,Finance,62000.000000


## 5. CTE with customer totals

In [6]:
pd.read_sql_query('WITH customer_totals AS (SELECT customer_id, SUM(amount) AS total_spend FROM orders GROUP BY customer_id) SELECT c.customer_name, COALESCE(t.total_spend, 0) AS total_spend FROM customers c LEFT JOIN customer_totals t ON t.customer_id = c.customer_id', connection)

,customer_name,total_spend
0,Customer A,375.0
1,Customer B,780.0
2,Customer C,80.0
3,Customer D,0.0
4,Customer E,320.0
5,Customer F,0.0
6,Customer G,0.0


## 6. Recursive CTE: integers 1 through 10

In [7]:
pd.read_sql_query('WITH RECURSIVE numbers(n) AS (SELECT 1 UNION ALL SELECT n + 1 FROM numbers WHERE n < 10) SELECT n FROM numbers', connection)

,n
0,1
1,2
2,3
3,4
4,5
5,6
6,7
7,8
8,9
9,10


## 7. Recursive CTE: multiples of 3

In [8]:
pd.read_sql_query('WITH RECURSIVE multiples(n) AS (SELECT 3 UNION ALL SELECT n + 3 FROM multiples WHERE n < 30) SELECT n FROM multiples', connection)

,n
0,3
1,6
2,9
3,12
4,15
5,18
6,21
7,24
8,27
9,30


## 8. Running revenue using a CTE

In [9]:
pd.read_sql_query('WITH monthly_revenue AS (SELECT substr(order_date, 1, 7) AS month, SUM(amount) AS revenue FROM orders GROUP BY substr(order_date, 1, 7)) SELECT month, revenue, SUM(revenue) OVER (ORDER BY month) AS running_revenue FROM monthly_revenue ORDER BY month', connection)

,month,revenue,running_revenue
0,2026-01,975.0,975.0
1,2026-02,580.0,1555.0


## 9. Rank employees within each department

In [10]:
pd.read_sql_query('WITH ranked AS (SELECT employee_name, department_id, salary, RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) AS salary_rank FROM employees WHERE salary IS NOT NULL) SELECT * FROM ranked ORDER BY department_id, salary_rank', connection)

,employee_name,department_id,salary,salary_rank
0,Asha,1,72000.0,1
1,Ravi,1,68000.0,2
2,Meera,1,61000.0,3
3,Kabir,2,85000.0,1
4,Neha,2,52000.0,2
5,Arjun,2,48000.0,3
6,Isha,3,78000.0,1
7,Dev,3,42000.0,2
8,Tara,4,65000.0,1
9,Zoya,4,63000.0,2


## 10. Departments with no employees

In [11]:
pd.read_sql_query('WITH employee_departments AS (SELECT DISTINCT department_id FROM employees) SELECT d.department_name FROM departments d LEFT JOIN employee_departments e ON e.department_id = d.department_id WHERE e.department_id IS NULL', connection)

,department_name
0,Research


## 11. Payroll above average department payroll

In [12]:
pd.read_sql_query('WITH payroll AS (SELECT department_id, SUM(salary) AS total_payroll FROM employees GROUP BY department_id), average_payroll AS (SELECT AVG(total_payroll) AS avg_payroll FROM payroll) SELECT p.department_id, p.total_payroll, a.avg_payroll FROM payroll p CROSS JOIN average_payroll a WHERE p.total_payroll > a.avg_payroll', connection)

,department_id,total_payroll,avg_payroll
0,1,201000.0,173000.0
1,2,185000.0,173000.0
2,4,186000.0,173000.0


## Key takeaways

- CTEs are named query results scoped to one statement.
- Multiple CTEs are separated by commas.
- Recursive CTEs need a base case and a termination condition.
- CTEs improve readability but do not guarantee better performance.